In [ ]:
print("HELLO WORLD")

HELLO WORLD


In [14]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import jax
import jax.numpy as jnp
from flax import linen as nn

import wavefunctions, trajectory, optimization, folx_hamiltonian
import local_optimization

import importlib

In [3]:
r_ws = 20
N = 56
tau = 1.25
dim = 2
numKpoints = 169

walkers = 1024

eqSteps      = 2000
trainSteps   = 1000
trainEqSteps = 25
evalSteps    = 1000

NUp = N // 2
NDown = N - NUp
spins = ( NUp , NDown )

lattice = wavefunctions.computeLattice(
    N, r_ws, dim, basis_matrix=jnp.array([[7,0],[0,4*jnp.sqrt(3)]])
)
kpoints = wavefunctions.genKpoints(numKpoints, lattice, dim)

( acceptMin , acceptMax ) = ( 0.55 , 0.65 )

T = 1000
diagonalShift = 1e-3
mu = 0.9

eta0 = 1e-2

In [4]:
class TestWavefunction(wavefunctions.Wavefunction):
    """
    TODO
    """
    spins : (int,int)
    dim : int
    lattice : jnp.ndarray
    kpoints : jnp.ndarray
    upCoeffs : jnp.ndarray
    downCoeffs : jnp.ndarray
    hiddenFeatures : int

    def setup(self):
        
        self.slaterUp = wavefunctions.LogMPSlater(
            self.spins[0], self.dim, self.kpoints, self.upCoeffs
        )
        self.slaterDown = wavefunctions.LogMPSlater(
            self.spins[1], self.dim, self.kpoints, self.downCoeffs
        )
        self.CYJastrow = wavefunctions.LogCYJastrow(self.spins, self.lattice)

        self.weights1_nb = nn.Dense(self.hiddenFeatures)
        self.weights2_nb = nn.Dense(1)

        self.linear1_nb = nn.Dense(self.hiddenFeatures)
        self.linear2_nb = nn.Dense(self.hiddenFeatures)
        
        self.jastrow1_nb = nn.Dense(self.hiddenFeatures)
        self.jastrow2_nb = nn.Dense(
            1, kernel_init=nn.initializers.zeros, bias_init=nn.initializers.zeros
        )
        
        self.backflow1_nb = nn.Dense(self.hiddenFeatures)
        self.backflow2_nb = nn.Dense(
            self.dim, kernel_init=nn.initializers.zeros, bias_init=nn.initializers.zeros
        )

    def __call__(self, rs):

        N = sum(self.spins)

        eye_mask = jnp.eye(N, dtype=bool)[:,:,None]
        v_ij = wavefunctions.generateFeatures(rs, self.spins, self.lattice)   # (N,N,2*dim+2)

        g_ij = nn.sigmoid(
            self.weights2_nb(nn.swish(self.weights1_nb(v_ij)))
        )                                                       # (N,N,1)
        h_ij = self.linear2_nb(nn.swish(self.linear1_nb(v_ij))) # (N,N,d)

        h_i = jnp.sum(
            jnp.where(~eye_mask, g_ij * h_ij, 0.0), axis=1
        ) / N                                                   # (N,d)

        jastrow_nb = jnp.sum(
            self.jastrow2_nb(nn.swish(self.jastrow1_nb(h_i)))
        ) / N

        backflow_nb = self.backflow2_nb(nn.swish(self.backflow1_nb(h_i))) # (N,dim)

        xs = rs + backflow_nb
        
        CYJastrow = self.CYJastrow(rs)
        slaterUp = self.slaterUp(xs[:self.spins[0],:])
        slaterDown = self.slaterDown(xs[self.spins[0]:,:])
        
        return slaterUp + slaterDown + CYJastrow + jastrow_nb

In [28]:
importlib.reload(local_optimization)

rng = jax.random.PRNGKey(558)
rng, rs_rng, init_rng = jax.random.split(rng, 3)

centers = trajectory.generateStripedAFM(spins, lattice, dim, gridShape=(7,4))
(upCenters,downCenters) = centers[:spins[0]], centers[spins[0]:]
rs = centers[None,:,:] + (r_ws / 10) * jax.random.normal(rs_rng, shape=(walkers,N,dim))

alpha = 1 / (r_ws ** 2.0)
upCoeffs = wavefunctions.generateGaussianMPCoeffs(kpoints, upCenters, alpha)
downCoeffs = wavefunctions.generateGaussianMPCoeffs(kpoints, downCenters, alpha)

wavefunction = TestWavefunction(spins, dim, lattice, kpoints, upCoeffs, downCoeffs, 32)
#wavefunction = wavefunctions.LogSlaterCYJastrow(spins, dim, lattice, kpoints)
mala = trajectory.MALAUpdater(wavefunction, r_ws)
if dim == 2:
    localEnergy = folx_hamiltonian.LocalEnergyUEG2D(wavefunction, lattice, sparsity_threshold=6)
elif dim == 3:
    localEnergy = folx_hamiltonian.LocalEnergyUEG(wavefunction, lattice)
optimizer = optimization.StochasticReconfigurationMomentum(
    wavefunction, localEnergy, mode='normal'
)

parameters = wavefunction.initBatch(init_rng, rs)

cyParametersPaths = [
    ('params', 'CYJastrow')
]
staticParametersPaths = [
    ('params', 'slaterUp'),
    ('params', 'slaterDown'),
]

allParametersMask = local_optimization.castFloatAsPytree(1.0, parameters)

dynamicParametersMask = local_optimization.castFloatAsPytree(1.0, parameters)
for path in staticParametersPaths:
    dynamicParametersMask = local_optimization.replaceSubtreeLeaves(dynamicParametersMask, path, 0.0)

staticParameterMask = local_optimization.castFloatAsPytree(0.0, parameters)
for path in staticParametersPaths:
    staticParameterMask = local_optimization.replaceSubtreeLeaves(staticParameterMask, path, 1.0)
for path in cyParametersPaths:
    staticParameterMask = local_optimization.replaceSubtreeLeaves(staticParameterMask, path, 1.0)

print(np.sum(local_optimization.flatten(allParametersMask)))
print(local_optimization.flatten(allParametersMask).shape)

print(np.sum(local_optimization.flatten(dynamicParametersMask)))
print(local_optimization.flatten(dynamicParametersMask).shape)

print(np.sum(local_optimization.flatten(staticParameterMask)))
print(local_optimization.flatten(staticParameterMask).shape)

print(rs.shape, kpoints.shape)

raise Exception("STOP")

( _ , _ , newParameters , _ ) = optimizer(
    parameters, rs, eta0, diagonalShift, 0.0, 0.0
)

13214.0
(13214,)
3750.0
(13214,)
9466.0
(13214,)
(1024, 56, 2) (169, 2)


Exception: STOP

In [7]:
print(parameters['params'].keys())

dict_keys(['weights1_nb', 'weights2_nb', 'linear1_nb', 'linear2_nb', 'jastrow1_nb', 'jastrow2_nb', 'backflow1_nb', 'backflow2_nb', 'CYJastrow', 'slaterUp', 'slaterDown'])
